# 类与类型检查

学习目标：能区分类的实例与构造侧，使用初始化、访问修饰符和继承契约，并理解实际生成代码。

前置知识：JavaScript 类、继承、字段、私有元素与 this；TypeScript 结构类型、泛型与收窄。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict。本章附加选项：noImplicitOverride=true、useDefineForClassFields=true，含义见对应知识点。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/10-classes/。

1. [main.ts](scripts/10-classes/main.ts)：按正文顺序组织的正常示例，片段依赖同文件前文定义。
2. [type-errors.ts](scripts/10-classes/type-errors.ts)：与正常示例隔离的类型反例，不生成或执行 JavaScript。
3. [tsconfig.json](scripts/10-classes/tsconfig.json)、[tsconfig.errors.json](scripts/10-classes/tsconfig.errors.json)：分别明确正常与反例文件范围。
4. [private-runtime-error.ts](scripts/10-classes/private-runtime-error.ts)：单独说明和执行的配套边界示例。



Step 1：检查正常项目的类型。

```bash
npm run check:10
```

Step 2：生成正常项目的 JavaScript。

```bash
npm run build:10
```

Step 3：运行正常示例。

```bash
npm run run:10
```

Step 4：检查下文独立列出的类型反例。

```bash
npm run errors:10
# 预期非零退出；按反例注释逐行核对具体错误，不运行 type-errors.ts。
```

正常配置只包含上面列出的正常与独立运行示例，生成文件位于 .build/10-classes/。错误配置继承正常选项，改用 type-errors.ts 并开启 noEmit。

## 1 实例类型与构造函数类型

类名在类型位置通常表示实例类型，在值位置表示构造函数。typeof Note 位于类型位置时，取得构造函数值的静态类型，包含构造签名和静态成员；它不是一个 Note 实例。

只需接收“能构造 Note 的值”时，可以用 new (title: string) =&gt; Note，而不要求全部静态成员。实例兼容通常按实例成员判断；构造函数和静态成员不会自动变成实例字段。

```typescript
export {};
class Note {
  static category = "学习";
  title: string;
  constructor(title: string) { this.title = title; }
}
const instance: Note = new Note("类");
const creator: typeof Note = Note;
const factory: new (title: string) => Note = creator;
console.log(instance.title, creator.category, new factory("构造").title); // 类 学习 构造
```

以下片段来自独立的 type-errors.ts：

```typescript
class InstanceOnly { title = "类"; }
const constructorAsInstance: InstanceOnly = InstanceOnly; // 构造函数值没有实例的 title 字段。
```

## 2 字段与严格初始化

strict 包含 strictPropertyInitialization，要求非可选字段在声明处或构造函数中得到明确初始化。类型允许 undefined 的字段可表达“尚未提供”，读取时相应检查。若初始化委托给普通方法，编译器不会通用地追踪它来证明构造完成。

字段后的 ! 是确定赋值断言，只向检查器作承诺，不会生成初始化代码。日常优先直接初始化。本章显式使用 useDefineForClassFields；在当前目标下保留标准类字段语义，不能把字段声明总是理解成简单调用继承 setter 的赋值。

```typescript
class Draft {
  title: string;
  pages = 0;
  subtitle: string | undefined;
  constructor(title: string) { this.title = title; }
}
const draft = new Draft("初始化");
console.log(draft.title, draft.pages, draft.subtitle); // 初始化 0 undefined
```

以下片段来自独立的 type-errors.ts：

```typescript
class Uninitialized { title: string; } // 严格初始化检查：没有初始值且构造函数未赋值。
```

## 3 public、private、protected 与 readonly

public 是默认可见性；private 只允许声明类内部按正常属性访问方式使用；protected 允许类内部和派生类使用，但不是任意外部对象都能访问。readonly 字段可在声明或本类构造函数中设置，之后不能通过该字段赋值。

这些 TypeScript 修饰符不会把普通属性变成 JavaScript 私有元素。readonly 也不递归冻结对象；有 private 或 protected 成员的类参与兼容比较时，还要考虑成员是否来自同一声明来源，不能只看名称与类型相同。

```typescript
class Entry {
  public title: string;
  private revision = 0;
  protected prefix = "课程:";
  readonly id: number;
  constructor(title: string, id: number) { this.title = title; this.id = id; }
  revise(): number { return ++this.revision; }
}
class PublishedEntry extends Entry {
  label(): string { return this.prefix + this.title; }
}
const entry = new PublishedEntry("类", 1);
console.log(entry.label(), entry.revise(), entry.id, Object.keys(entry).includes("revision")); // 课程:类 1 1 true
```

以下片段来自独立的 type-errors.ts：

```typescript
class ProtectedEntry {
  private revision = 0;
  protected prefix = "课";
  readonly id = 1;
}
const hidden = new ProtectedEntry();
hidden.revision; // private 不能在声明类外正常访问。
hidden.prefix; // protected 不能在任意外部代码访问。
hidden.id = 2; // 构造完成后不能给只读字段重新赋值。
class FirstPrivate { private id = 1; }
class SecondPrivate { private id = 1; }
const separateOrigin: FirstPrivate = new SecondPrivate(); // private 成员来自不同声明。
```

## 4 参数属性确实生成字段

构造参数加上 public、private、protected 或 readonly 等修饰符，就成为参数属性（parameter property）：编译器为实例创建同名字段并保存参数值。普通构造参数没有这种效果。

下面 summary 使用的 title 和 minutes 是实例字段；strict 检查其类型，参数属性转换负责实际赋值。构建后在 .build/10-classes/main.js 中可以看到参数属性对应的字段与构造函数赋值，而修饰符与类型标注不再保留。

```typescript
class Lesson {
  constructor(public readonly title: string, private minutes: number) {}
  summary(): string { return this.title + ":" + this.minutes; }
}
const lesson = new Lesson("参数属性", 10);
console.log(lesson.summary(), Object.keys(lesson).join(",")); // 参数属性:10 title,minutes
```

## 5 abstract、implements 与 extends

abstract 类不能由 TypeScript 代码直接实例化，抽象成员只描述派生类需要实现的接口。implements 检查类的实例结构是否满足契约，不会复制实现，也不为未标注的方法参数自动提供上下文类型。

extends 建立实际继承关系，派生构造函数在使用 this 前必须调用 super。下面具体类既继承抽象基类，又满足 Renderable。工厂要求非抽象构造签名，因此可以真正创建返回实例；typeof 抽象基类则可能包含抽象构造限制。

```typescript
interface Renderable { render(): string; }
abstract class DocumentBase {
  abstract render(): string;
  describe(): string { return "文档:" + this.render(); }
}
class TextDocument extends DocumentBase implements Renderable {
  constructor(public text: string) { super(); }
  override render(): string { return this.text; }
}
function buildDocument(creator: new (text: string) => DocumentBase): DocumentBase {
  return new creator("抽象类");
}
console.log(buildDocument(TextDocument).describe()); // 文档:抽象类
```

以下片段来自独立的 type-errors.ts：

```typescript
abstract class AbstractTask { abstract run(): string; }
new AbstractTask(); // 抽象类不能直接实例化。
class MissingTask extends AbstractTask {} // 非抽象派生类缺少 run 实现。
interface NeedsRun { run(value: string): string; }
class MissingImplementation implements NeedsRun {} // implements 不会生成 run。
class Uninferred implements NeedsRun {
  run(value) { return value; } // implements 不为这里的参数提供类型，strict 下隐式 any 被拒绝。
}
```

## 6 override 与派生契约

本章额外开启 noImplicitOverride，覆盖基类已有成员时显式写 override；如果基类成员被删除或改名，override 也能发现派生代码失去对应关系。

覆盖后的成员仍必须兼容基类使用方式，不能要求基类调用者原本不需要传入的必需参数。override 不自动调用基类实现；需要复用行为时显式调用 super。

```typescript
class BaseLabel { label(): string { return "基础"; } }
class DetailedLabel extends BaseLabel {
  override label(): string { return super.label() + "/扩展"; }
}
console.log(new DetailedLabel().label()); // 基础/扩展
```

以下片段来自独立的 type-errors.ts：

```typescript
class OverrideBase { label(): string { return "基础"; } }
class ForgottenOverride extends OverrideBase {
  label(): string { return "派生"; } // noImplicitOverride 要求显式 override。
}
class WrongOverride extends OverrideBase {
  override label(required: number): string { return String(required); } // 基类允许无参数调用。
}
```

## 7 泛型类与 this 类型守卫

Box&lt;T&gt; 的 T 表示当前实例保存的值类型。类的静态侧只有一份运行时对象，因此 static 成员不能引用实例类型参数 T。需要静态泛型函数时，为该函数另声明自己的类型参数。

返回 this 的方法保留派生类的具体类型，适合链式调用。this is ... 方法则根据实际检查收窄当前对象；下面用公开 value 与 undefined 判断建立依据。守卫只反映检查位置的状态，不应在随后修改对象后仍假定内容永远不空。

```typescript
class Box<T> {
  value: T | undefined;
  set(value: T): this { this.value = value; return this; }
  hasValue(): this is this & { value: T } { return this.value !== undefined; }
}
class TextBox extends Box<string> { label = "文本"; }
const box = new TextBox().set("TS");
const size = box.hasValue() ? box.value.length : 0;
const empty = new Box<number>();
console.log(box.label, size, empty.hasValue()); // 文本 2 false
```

以下片段来自独立的 type-errors.ts：

```typescript
class StaticGeneric<T> { static initial: T; } // 静态成员不能引用类的实例类型参数 T。
```

## 8 TypeScript private 与 JavaScript #

TypeScript private 主要是检查期的访问限制，输出中的普通属性仍可被 JavaScript 观察到。JavaScript # 私有元素由运行时检查私有身份；同名普通字符串属性不等于该私有元素。

下例只打印普通属性名，并在正常实例上读取 #secret。独立的 private-runtime-error.ts 用错误接收者调用读取方法；即使调用表达式的类型允许，JavaScript 仍拒绝访问没有相应私有身份的对象。

```typescript
class Vault {
  private ordinary = 1;
  #secret = 2;
  read(): number { return this.#secret; }
}
const vault = new Vault();
console.log(Object.keys(vault).join(","), vault.read()); // ordinary 2
```

## 9 混入复用一组实例能力

混入（mixin）可以写成接收基类、返回继承该基类的类表达式的函数。TBase 表示输入构造函数类型；示例中的 Constructor 使用任意参数列表，是这种通用混入构造签名的必要范围，不把业务成员改成 any。

这里的混入只添加一个私有布尔状态和方法，不依赖基类的特定成员。若混入需要基类某项能力，应约束构造后的实例结构。使用类表达式模式时避免在混入中声明 TypeScript private/protected 成员，可使用 JavaScript # 私有元素保存内部状态；不展开旧式手工复制原型方案。

```typescript
type Constructor = new (...args: any[]) => object;
function WithFlag<TBase extends Constructor>(Base: TBase) {
  return class extends Base {
    #enabled = false;
    enable(): void { this.#enabled = true; }
    isEnabled(): boolean { return this.#enabled; }
  };
}
const FlaggedNote = WithFlag(Note);
const flagged = new FlaggedNote("混入");
flagged.enable();
console.log(flagged.title, flagged.isEnabled()); // 混入 true
```

## 10 独立观察私有身份失败

以下为 private-runtime-error.ts 的完整内容。普通方法的签名没有显式约束 this 为 Secret，因此 call 的表达式能通过此处静态检查；方法体的 #value 访问仍由运行时检查。

```typescript
export {};
class Secret {
  #value = 1;
  read(): number { return this.#value; }
}
Secret.prototype.read.call({}); // TypeError：接收者没有 Secret 的私有身份。
```

Step 1：在已执行 build:10 后单独运行反例。

```bash
node .build/10-classes/private-runtime-error.js
# 预期退出码 1；TypeError，消息包含 Cannot read private member #value。
```

## 本章小结

类同时提供运行时构造函数和静态实例类型。初始化与继承契约需要实际实现，implements 不生成行为；参数属性会生成赋值，private 和 readonly 不等于 JavaScript # 私有身份。泛型、this 与混入服务于明确的实例关系。

## 练习

1. 实现只读编号、可变标题的泛型记录类，初始化后修改编号应产生诊断，更新标题应通过；构建后确认对应普通属性仍在实例中。
2. 为抽象任务类实现两个具体派生类，用非抽象构造签名创建并运行；抽象基类本身不能传入工厂。
3. 给 Box 加清空方法，分别检查空、设置后、再次清空三种状态；hasValue 的结果应与实际值一致。
4. 给需要动态 this 的读取方法补充显式 this 参数，确认错误接收者调用变成类型错误；对正确实例的行为保持不变。

### 提示

1. 让类型参数 T 表示编号类型，构造参数可写 public readonly id: T、public title: string。
2. 工厂参数用 new (...) =&gt; 基类实例类型，不用 abstract new。
3. clear 只需将 this.value 设为 undefined；每次改变后重新调用 hasValue。
4. 在 Secret.read 签名中写 this: Secret，再分别调用正确实例与空对象。


### 参考解析

1. 编号的只读限制属于检查期；例如字符串编号不能重新赋值，而标题可更新。生成的 JavaScript 仍有普通 id 和 title 字段。
2. 两个具体类分别实现 run 后可传入工厂；抽象构造函数不满足能直接 new 的签名。
3. 新建、set 后、clear 后应依次为 false、true、false；不可沿用修改前的分支结论。
4. 正确实例仍返回私有值；read.call({}) 的静态实参不满足 Secret。this 标注本身被擦除，运行时 # 身份检查仍然存在。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [Classes：成员、继承、可见性、泛型、this、参数属性与抽象构造](https://www.typescriptlang.org/docs/handbook/2/classes.html)；[Mixins：类表达式与约束](https://www.typescriptlang.org/docs/handbook/mixins.html)；[strictPropertyInitialization](https://www.typescriptlang.org/tsconfig/strictPropertyInitialization.html)；[noImplicitOverride](https://www.typescriptlang.org/tsconfig/noImplicitOverride.html)；[useDefineForClassFields](https://www.typescriptlang.org/tsconfig/useDefineForClassFields.html)；[3.8：ECMAScript 私有字段与 TypeScript private](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-3-8.html#ecmascript-private-fields)；[4.3：override](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-4-3.html#override-and-the---noimplicitoverride-flag)；[Type Compatibility：private 与 protected 来源](https://www.typescriptlang.org/docs/handbook/type-compatibility.html#private-and-protected-members-in-classes)。 |
| npm 官方文档 | [npm run（v11）](https://docs.npmjs.com/cli/v11/commands/npm-run/)：从本技术目录运行已配置脚本，并解析本地工具。 |
